# 08 — Production & Performance (vectorization, memory, chunking, error handling)

Maps to **MODULE-08a / 08b**.

Docs: `datasets/README.md` (Phases 3 & 6)

In [ ]:
import time
import pandas as pd

orders = pd.read_csv('datasets/raw/orders.csv')

### 1. Vectorization vs loops vs apply

In [ ]:
start = time.time()
result = []
for i in range(len(orders)):
    result.append(orders['subtotal'].iloc[i] + orders['tax'].iloc[i])
loop_time = time.time() - start

start = time.time()
orders.apply(lambda r: r['subtotal'] + r['tax'], axis=1)
apply_time = time.time() - start

start = time.time()
orders['subtotal'] + orders['tax']
vec_time = time.time() - start

print(f'loop={loop_time:.4f}s  apply={apply_time:.4f}s  vectorized={vec_time:.4f}s')

### 2. Memory usage & downcasting

In [ ]:
orders.memory_usage(deep=True).sum() / 1024  # KB

In [ ]:
def downcast(df):
    for c in df.select_dtypes(include='int64'):
        df[c] = pd.to_numeric(df[c], downcast='integer')
    for c in df.select_dtypes(include='float64'):
        df[c] = pd.to_numeric(df[c], downcast='float')
    return df

orders_opt = downcast(orders.copy())
orders_opt.memory_usage(deep=True).sum() / 1024  # KB

### 3. Categorical for low-cardinality strings

In [ ]:
customers = pd.read_csv('datasets/raw/customers.csv')
before = customers['membership'].memory_usage(deep=True)
customers['membership'] = customers['membership'].astype('category')
after = customers['membership'].memory_usage(deep=True)
print(f'membership memory: {before} -> {after} bytes')

### 4. Chunked loading

In [ ]:
# For very large files (e.g. datasets/transactions.json, 122 MB — regenerate
# with `python generate_datasets.py`), process in chunks. Demonstrate on orders.csv:
chunks = []
for chunk in pd.read_csv('datasets/raw/orders.csv', chunksize=1000):
    chunks.append(chunk[chunk['status'] == 'completed'])
done = pd.concat(chunks)
done.shape

### 5. Error handling & validation

In [ ]:
def safe_load(path):
    from pathlib import Path
    if not Path(path).exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    if df.empty:
        raise ValueError(f'{path} is empty')
    return df

safe_load('datasets/raw/orders.csv').shape

### 6. Copy-on-Write

In [ ]:
pd.options.mode.copy_on_write = True